# 전처리

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# cmapss_comparison.py
# 목적: Linear / LightGBM / LSTM / Transformer 4개 모델 비교
# ════════════════════════════════════════════════════════════════════════════

# ── 0. 라이브러리 ─────────────────────────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
import mlflow
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.cluster import KMeans
from lightgbm import LGBMRegressor
from lightgbm import early_stopping as lgb_early_stopping

warnings.filterwarnings("ignore")

COLS    = ["unit", "cycle", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]
OP_COLS = ["op1", "op2", "op3"]
WINDOW  = 30   # DL 시퀀스 윈도우 크기


# ════════════════════════════════════════════════════════════════════════════
# 1. 공통 전처리  (모든 모델 공유)
# ════════════════════════════════════════════════════════════════════════════
def load_cmapss(dataset: str, data_dir: str = "CMAPSSData"):
    kw = dict(sep=r"\s+", header=None, names=COLS, engine="python")
    train = pd.read_csv(os.path.join(data_dir, f"train_{dataset}.txt"), **kw)
    test  = pd.read_csv(os.path.join(data_dir, f"test_{dataset}.txt"),  **kw)
    rul   = pd.read_csv(os.path.join(data_dir, f"RUL_{dataset}.txt"),
                        sep=r"\s+", header=None, names=["RUL"], engine="python")
    return train, test, rul


def select_sensors(train_raw: pd.DataFrame) -> list:
    """분산 거의 없는 센서 제거 (정보량 없음)"""
    stds = train_raw[[f"s{i}" for i in range(1, 22)]].std()
    return stds[stds > 0.01].index.tolist()


def apply_piecewise_rul(df: pd.DataFrame, dataset: str) -> pd.DataFrame:
    """초기 안정 구간 RUL 상한선 적용"""
    cap = 135 if dataset in ["FD002", "FD004"] else 125
    mc  = df.groupby("unit")["cycle"].max().rename("max_cycle")
    df  = df.join(mc, on="unit")
    df["RUL"] = (df["max_cycle"] - df["cycle"]).clip(upper=cap)
    return df.drop("max_cycle", axis=1)


def common_preprocess(dataset: str, data_dir: str = "CMAPSSData"):
    """
    공통 전처리 반환값:
      train_df, test_df  - 정렬 + cycle_norm + RUL(train only)
      rul_df             - 테스트 정답
      sensors            - 유효 센서 리스트
    """
    train_raw, test_raw, rul_df = load_cmapss(dataset, data_dir)
    sensors = select_sensors(train_raw)

    train_df = train_raw.sort_values(["unit", "cycle"]).reset_index(drop=True)
    test_df  = test_raw.sort_values(["unit", "cycle"]).reset_index(drop=True)

    for df in [train_df, test_df]:
        df["cycle_norm"] = df.groupby("unit")["cycle"].transform(lambda x: x / x.max())

    train_df = apply_piecewise_rul(train_df, dataset)
    return train_df, test_df, rul_df, sensors


# ════════════════════════════════════════════════════════════════════════════
# 2. ML 브랜치  (Linear, LightGBM)
# ════════════════════════════════════════════════════════════════════════════
def build_ml_features(df: pd.DataFrame, sensors: list, add_diff: bool = True):
    """스무딩 + diff 피처 빌드 (LGB만 add_diff=True)"""
    df = df.copy()
    grouped = df.groupby("unit")
    for s in sensors:
        s_smooth = grouped[s].transform(
            lambda x: x.rolling(window=5, min_periods=5).mean()
        ).bfill().ffill()
        if add_diff:
            df[f"{s}_diff"] = s_smooth.groupby(df["unit"]).diff().fillna(0)
        df[s] = s_smooth

    features = OP_COLS + sensors + ["cycle_norm"]
    if add_diff:
        features += [f"{s}_diff" for s in sensors]
    return df, features


def select_features_once(df: pd.DataFrame, candidate_features: list, top_n: int = 12) -> list:
    """LGB importance 기반 피처 선택 — fold마다 X, 전체 train으로 한 번만"""
    tmp = LGBMRegressor(n_estimators=300, random_state=42, verbose=-1)
    tmp.fit(df[candidate_features], df["RUL"])
    importance = pd.Series(tmp.feature_importances_, index=candidate_features).sort_values(ascending=False)
    must   = OP_COLS + ["cycle_norm"]
    others = [f for f in importance.index if f not in must]
    return must + others[:(top_n - len(must))]


def run_ml_model(model_name: str, dataset: str, data_dir: str = "CMAPSSData"):
    train_df, test_df, rul_df, sensors = common_preprocess(dataset, data_dir)

    # 모델별 피처 빌드
    add_diff = (model_name == "lightgbm")
    train_df, candidate_features = build_ml_features(train_df, sensors, add_diff)
    test_df,  _                  = build_ml_features(test_df,  sensors, add_diff)

    # LGB 전용: KMeans 조건 분류
    if model_name == "lightgbm" and dataset in ["FD002", "FD004"]:
        km = KMeans(n_clusters=6, random_state=42)
        train_df["_cond"] = km.fit_predict(train_df[OP_COLS])
        test_df["_cond"]  = km.predict(test_df[OP_COLS])
    else:
        train_df["_cond"] = 0
        test_df["_cond"]  = 0

    # 피처 확정
    if model_name == "lightgbm":
        selected = select_features_once(train_df, candidate_features)
    else:
        selected = candidate_features  # Linear는 전체 사용

    gkf       = GroupKFold(n_splits=5)
    oof_preds = np.zeros(len(train_df))
    models_info = []

    for fold, (tr_idx, val_idx) in enumerate(
        gkf.split(train_df, train_df["RUL"], train_df["unit"])
    ):
        tr_fold  = train_df.iloc[tr_idx].copy()
        val_fold = train_df.iloc[val_idx].copy()

        # ── 스케일링 ──────────────────────────────────────────────────────
        if model_name == "lightgbm":
            # 조건별 스케일링 + Global Fallback
            g_sc = StandardScaler().fit(tr_fold[selected])
            scalers = {}
            for c in sorted(set(tr_fold["_cond"]) | set(val_fold["_cond"])):
                c_tr  = tr_fold["_cond"]  == c
                c_val = val_fold["_cond"] == c
                if c_tr.any():
                    sc = StandardScaler()
                    sc.fit(tr_fold.loc[c_tr, selected])
                    tr_fold.loc[c_tr, selected]   = sc.transform(tr_fold.loc[c_tr, selected])
                    if c_val.any():
                        val_fold.loc[c_val, selected] = sc.transform(val_fold.loc[c_val, selected])
                    scalers[c] = sc
                elif c_val.any():
                    val_fold.loc[c_val, selected] = g_sc.transform(val_fold.loc[c_val, selected])
        else:
            # Linear: 단순 글로벌 스케일링
            sc = StandardScaler()
            tr_fold[selected]  = sc.fit_transform(tr_fold[selected])
            val_fold[selected] = sc.transform(val_fold[selected])
            scalers = {0: sc}
            g_sc = sc

        # ── 모델 학습 ─────────────────────────────────────────────────────
        if model_name == "lightgbm":
            model = LGBMRegressor(
                n_estimators=1000, learning_rate=0.03, num_leaves=31,
                random_state=42, verbose=-1
            )
            model.fit(
                tr_fold[selected], tr_fold["RUL"],
                eval_set=[(val_fold[selected], val_fold["RUL"])],
                callbacks=[lgb_early_stopping(50, verbose=False)],
            )
        else:
            model = Ridge(alpha=1.0)
            model.fit(tr_fold[selected], tr_fold["RUL"])

        oof_preds[val_idx] = model.predict(val_fold[selected])
        models_info.append((model, selected, scalers, g_sc))
        print(f"  Fold {fold+1} 완료")

    oof_rmse = np.sqrt(mean_squared_error(train_df["RUL"], oof_preds))

    # ── 테스트 예측 ───────────────────────────────────────────────────────
    test_last = test_df.groupby("unit").tail(1).index
    y_test    = rul_df["RUL"].values
    preds     = []

    for model, feats, scalers, g_sc in models_info:
        tmp = test_df.copy()
        for c in tmp["_cond"].unique():
            idx = tmp["_cond"] == c
            tmp.loc[idx, feats] = scalers.get(c, g_sc).transform(tmp.loc[idx, feats])
        preds.append(model.predict(tmp.loc[test_last, feats]))

    test_rmse = np.sqrt(mean_squared_error(y_test, np.mean(preds, axis=0)))
    return oof_rmse, test_rmse


# ════════════════════════════════════════════════════════════════════════════
# 3. DL 브랜치  (LSTM, Transformer)
# ════════════════════════════════════════════════════════════════════════════
def make_sequences(df: pd.DataFrame, features: list, window: int = WINDOW):
    """슬라이딩 윈도우 → (N, window, features)"""
    X, y, groups = [], [], []
    for unit, grp in df.groupby("unit"):
        vals = grp[features].values.astype(np.float32)
        rul  = grp["RUL"].values.astype(np.float32)
        for i in range(len(grp) - window + 1):
            X.append(vals[i:i+window])
            y.append(rul[i+window-1])
            groups.append(unit)
    return np.array(X), np.array(y), np.array(groups)


def make_test_sequences(df: pd.DataFrame, features: list, window: int = WINDOW):
    """테스트셋: 각 유닛의 마지막 window 추출 (짧으면 zero-pad)"""
    out = []
    for _, grp in df.groupby("unit"):
        vals = grp[features].values.astype(np.float32)
        if len(vals) >= window:
            out.append(vals[-window:])
        else:
            pad = np.zeros((window - len(vals), len(features)), dtype=np.float32)
            out.append(np.vstack([pad, vals]))
    return np.array(out)


class LSTMModel(nn.Module):
    def __init__(self, input_size: int, hidden: int = 64, layers: int = 2, dropout: float = 0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, layers, batch_first=True, dropout=dropout)
        self.fc   = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)


class TransformerModel(nn.Module):
    def __init__(self, input_size: int, d_model: int = 64, nhead: int = 4,
                 layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.proj    = nn.Linear(input_size, d_model)
        enc_layer    = nn.TransformerEncoderLayer(
            d_model, nhead, dim_feedforward=128, dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, layers)
        self.fc      = nn.Linear(d_model, 1)

    def forward(self, x):
        x = self.proj(x)
        x = self.encoder(x)
        return self.fc(x[:, -1, :]).squeeze(-1)


def train_dl_fold(model, X_tr, y_tr, X_val, y_val,
                  device, epochs=100, batch_size=256, patience=10):
    model.to(device)
    opt       = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()
    loader    = DataLoader(
        TensorDataset(torch.tensor(X_tr).to(device), torch.tensor(y_tr).to(device)),
        batch_size=batch_size, shuffle=True
    )
    X_val_t = torch.tensor(X_val).to(device)
    y_val_t = torch.tensor(y_val).to(device)

    best_loss, best_state, no_improve = float("inf"), None, 0
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            criterion(model(xb), yb).backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val_t), y_val_t).item()

        if val_loss < best_loss:
            best_loss  = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    model.load_state_dict(best_state)
    return model


def run_dl_model(model_name: str, dataset: str, data_dir: str = "CMAPSSData",
                 window: int = WINDOW):
    train_df, test_df, rul_df, sensors = common_preprocess(dataset, data_dir)
    features = sensors + ["cycle_norm"]   # diff 없이 raw 센서만

    # 단순 글로벌 스케일링 (DL은 직접 패턴 학습)
    sc = StandardScaler()
    train_df[features] = sc.fit_transform(train_df[features])
    test_df[features]  = sc.transform(test_df[features])

    X, y, groups = make_sequences(train_df, features, window)
    device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    input_size   = X.shape[2]

    gkf       = GroupKFold(n_splits=5)
    oof_preds = np.full(len(X), np.nan)
    fold_models = []

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        model = LSTMModel(input_size) if model_name == "lstm" else TransformerModel(input_size)
        model = train_dl_fold(model, X[tr_idx], y[tr_idx], X[val_idx], y[val_idx], device)

        model.eval()
        with torch.no_grad():
            oof_preds[val_idx] = model(torch.tensor(X[val_idx]).to(device)).cpu().numpy()

        fold_models.append(model)
        print(f"  Fold {fold+1} 완료")

    valid    = ~np.isnan(oof_preds)
    oof_rmse = np.sqrt(mean_squared_error(y[valid], oof_preds[valid]))

    # 테스트 예측
    X_test   = torch.tensor(make_test_sequences(test_df, features, window)).to(device)
    y_test   = rul_df["RUL"].values
    preds    = []
    for m in fold_models:
        m.eval()
        with torch.no_grad():
            preds.append(m(X_test).cpu().numpy())

    test_rmse = np.sqrt(mean_squared_error(y_test, np.mean(preds, axis=0)))
    return oof_rmse, test_rmse


# ════════════════════════════════════════════════════════════════════════════
# 4. MLflow 실험 러너
# ════════════════════════════════════════════════════════════════════════════
ML_PARAMS = {
    "linear":   {"alpha": 1.0},
    "lightgbm": {"n_estimators": 1000, "lr": 0.03, "num_leaves": 31},
}
DL_PARAMS = {
    "lstm":        {"hidden": 64, "layers": 2, "window": WINDOW, "epochs": 100},
    "transformer": {"d_model": 64, "nhead": 4,  "window": WINDOW, "epochs": 100},
}
ML_MODELS = list(ML_PARAMS.keys())
DL_MODELS = list(DL_PARAMS.keys())


def run_experiment(model_name: str, dataset: str, data_dir: str = "CMAPSSData"):
    print(f"\n▶ [{dataset}] {model_name} 시작")
    params = {**(ML_PARAMS if model_name in ML_MODELS else DL_PARAMS)[model_name],
              "dataset": dataset}

    with mlflow.start_run(run_name=f"{model_name}_{dataset}"):
        mlflow.set_tags({"model": model_name, "dataset": dataset})
        mlflow.log_params(params)

        if model_name in ML_MODELS:
            oof_rmse, test_rmse = run_ml_model(model_name, dataset, data_dir)
        else:
            oof_rmse, test_rmse = run_dl_model(model_name, dataset, data_dir)

        mlflow.log_metrics({"oof_rmse": oof_rmse, "test_rmse": test_rmse})
        print(f"  OOF={oof_rmse:.4f}  TEST={test_rmse:.4f}")

    return {"model": model_name, "dataset": dataset,
            "oof_rmse": round(oof_rmse, 4), "test_rmse": round(test_rmse, 4)}


def run_all(datasets=None, data_dir: str = "CMAPSSData"):
    if datasets is None:
        datasets = ["FD001", "FD002", "FD003", "FD004"]

    mlflow.set_experiment("cmapss_model_comparison")
    results = [
        run_experiment(m, d, data_dir)
        for d in datasets
        for m in ML_MODELS + DL_MODELS
    ]

    pivot = pd.DataFrame(results).pivot_table(
        index="model", columns="dataset", values="test_rmse"
    )
    print("\n\n📊 최종 비교 결과 (TEST RMSE)\n", pivot.to_string())
    return pivot


# ════════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    run_all()